In [0]:
from pyspark.sql.functions import explode, col, current_timestamp

# Zaktualizowane ścieżki z właściwą nazwą katalogu (dbw_showcase)
RAW_HR_DIR = "/Volumes/dbw_showcase/default/showcase_raw/hr_batch"
RAW_PAYROLL_DIR = "/Volumes/dbw_showcase/default/showcase_raw/payroll_stream"
CHECKPOINT_PAYROLL = "/Volumes/dbw_showcase/default/showcase_raw/checkpoints_payroll"

# ==========================================
# 2. BATCH INGESTION (HR)
# ==========================================
print("⏳ Ładowanie danych HR (Batch)...")
hr_raw_df = spark.read.json(RAW_HR_DIR)

# Rozbijamy zagnieżdżoną listę 'employees' (rozwiązanie zadania z rekrutacji)
hr_bronze_df = hr_raw_df.select(
    col("dept_id"),
    col("department_name"),
    explode(col("employees")).alias("employee") 
).select(
    col("dept_id"),
    col("department_name"),
    col("employee.emp_id").alias("emp_id"),
    col("employee.name").alias("emp_name"),
    col("employee.city").alias("city"),
    col("employee.updated_at").alias("updated_at"),
    current_timestamp().alias("ingested_at") 
)

# Zapis do tabeli Unity Catalog w warstwie Bronze
hr_bronze_df.write.format("delta").mode("overwrite").saveAsTable("dbw_showcase.default.hr_bronze")
print("✅ Dane HR zapisane w warstwie Bronze jako tabela: dbw_showcase.default.hr_bronze")


# ==========================================
# 3. STREAMING INGESTION (Payroll) - Auto Loader
# ==========================================
print("⏳ Uruchamianie Auto Loadera dla logów Payroll...")

payroll_stream_df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", CHECKPOINT_PAYROLL + "_schema") 
    .load(RAW_PAYROLL_DIR)
    .withColumn("ingested_at", current_timestamp())
)

# Zapis strumieniowy do tabeli UC
query = (payroll_stream_df.writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT_PAYROLL)
    .trigger(availableNow=True) 
    .toTable("dbw_showcase.default.payroll_bronze")
)

query.awaitTermination()
print("✅ Dane Payroll załadowane do tabeli: dbw_showcase.default.payroll_bronze")